# 02. 하이퍼파라미터 튜닝 (한 번만)

C2(텍스트 + Marqo) 조건, seed 42로 후보 설정을 검증 주에서 비교한다. 끝나면 가장 좋은 설정을 `paper/configs/paper.yaml`의 `train.default`에 반영하고 `tuned: true`로 바꿔 커밋·푸시한다.

In [ ]:
# @title 0. 공통 준비 (모든 노트북 동일)
import sys
if any(m.startswith("msrs_paper") for m in sys.modules):
    raise RuntimeError("이전에 불러온 코드가 메모리에 남아 있습니다. 메뉴 '런타임 → 세션 다시 시작' 후 이 셀부터 다시 실행하세요.")
from google.colab import drive
drive.mount("/content/drive")

REPO_URL = "https://github.com/yuhwani/multimodal-recsys.git"  # upstream에 머지되면 3sunghynix 주소로 변경
BRANCH = "main"
!rm -rf /content/repo && git clone -q -b {BRANCH} {REPO_URL} /content/repo
!pip -q install -r /content/repo/paper/requirements.txt
!cd /content/repo && git log -1 --format="코드 커밋: %h %s (%cd)"

import sys
sys.path.insert(0, "/content/repo/paper")
from msrs_paper.config import load_all, Paths
cfg, exps, plan = load_all()
DRIVE_ROOT = cfg["drive_root"]  # 공유 폴더 바로가기 경로가 다르면 이 줄만 바꾼다
paths = Paths(DRIVE_ROOT).make_dirs()
print("드라이브 폴더:", paths.root)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# @title 1. 공유 준비물 확인 (체크섬이 다르면 멈춤)
from msrs_paper.utils import verify_manifest
m = verify_manifest(paths)
print(f"준비물 {len(m['files'])}개 일치 — 준비물 커밋 {m['git_commit']}")

In [ ]:
# @title 2. 후보 설정 비교 (중단돼도 다시 실행하면 이어서)
import pandas as pd
from msrs_paper import two_tower
rows = []
for hp in cfg["train"]["candidates"]:
    info = two_tower.run(cfg, exps, paths, "C2", 42, "main", "tuning", mode="tune", hp_name=hp)
    rows.append({"hp": hp, **cfg["train"]["candidates"][hp], "valid_recall@300": info["valid_recall@300"],
                 "train_seconds": info["train_seconds"]})
table = pd.DataFrame(rows).sort_values("valid_recall@300", ascending=False)
table.to_csv(paths.results / "tuning.csv", index=False)
table

1회 학습 시간(`train_seconds`)도 확인한다. 실험 A는 13회라서, 1회가 30분을 넘으면 seed 44 묶음(C0·C2·C3·C4)을 다른 사람에게 옮기는 것을 검토한다.